In [2]:
import wandb
import numpy as np
import json

def initialize_api():
    """Initialize the Weights & Biases API."""
    return wandb.Api()

def get_runs(api, username, project_name):
    """Retrieve all runs from the specified project."""
    project_path = f'{username}/{project_name}'
    return api.runs(project_path)

def collect_metrics(runs):
    """Group metrics by run names and ensure numerical values."""
    grouped_metrics = {}
    
    for run in runs:
        run_name = run.name if run.name else "Unnamed"
        
        # Collect metrics ensuring they are numerical
        metrics = {}
        for key, value in run.summary.items():
            if isinstance(value, (int, float)):
                metrics[key] = value
            else:
                try:
                    metrics[key] = float(value)
                except (TypeError, ValueError):
                    print(f"Skipping non-numeric metric '{key}' in run '{run_name}'")
                    continue
        
        # Group metrics by run name
        if run_name not in grouped_metrics:
            grouped_metrics[run_name] = []
        grouped_metrics[run_name].append(metrics)
    
    return grouped_metrics

def calculate_statistics(grouped_metrics, save = False, output_filename='metrics_summary.json'):
    """Calculate mean and standard deviation for each group and save results to a JSON file."""
    summary = {}

    for run_name, metrics_list in grouped_metrics.items():
        if len(metrics_list) != 3:
            raise ValueError(f"Warning: Run name '{run_name}' does not have exactly 3 runs (has {len(metrics_list)})")

        # Initialize dictionary to hold summary statistics for the current run name
        run_summary = {}

        # Collect all keys (metrics names) from the first run of the group
        keys = metrics_list[0].keys()
        metrics_summary = {key: {'values': []} for key in keys}

        # Gather all metric values
        for metrics in metrics_list:
            for key in keys:
                if key in metrics:
                    metrics_summary[key]['values'].append(metrics[key])

        # Calculate mean and std for each metric in the group
        for key, data in metrics_summary.items():
            values = np.array(data['values'])
            mean = np.mean(values)
            std = np.std(values)
            
            # Store the calculated mean and std in the run summary
            run_summary[key] = {'mean': mean, 'std': std}

        # Add the run summary to the overall summary
        summary[run_name] = run_summary

    # Write the summary to a JSON file
    if save:
        with open(output_filename, 'w') as f:
            json.dump(summary, f, indent=4)

    return summary

def print_results(results, ACCEPTED_KEYS):
    # Iterate over the results
    for run_name in sorted(results.keys()):
        metrics = results[run_name]
        print("-"*80)
        # Sort the metrics by the order in ACCEPTED_KEYS
        sorted_metrics = {k: metrics[k] for k in ACCEPTED_KEYS if k in metrics}
        
        # Print the sorted metrics
        for metric_name in sorted_metrics:
            values = sorted_metrics[metric_name]
            print(f"Run: {run_name}   |   {metric_name}: ({values['mean']:.5f}, {values['std']:.5f})")

def main():
    username = 'drigoni'  # Replace with your wandb username
    project_name = 'd4-hparams'

    # retrieve results from wandb
    api = initialize_api()
    runs = get_runs(api, username, project_name)
    grouped_metrics = collect_metrics(runs)

    # get all metrics
    all_keys = set()
    for run_metrics in grouped_metrics.values():
        all_keys.update(run_metrics[0].keys())
    print('List of all metrics:', sorted(all_keys))

    # calculate results average and std
    results = calculate_statistics(grouped_metrics)

    # print only accepted keys
    ACCEPTED_KEYS = ['test/molecular_validity', 'test/molecular_uniqueness', 'test/molecular_novelty', 
                     'test/bond_distance', 'test/bond_distance_per_class/single', 'test/bond_distance_per_class/double', 'test/bond_distance_per_class/triple',
                     'test/edge_types_distribution/single', 'test/edge_types_distribution/double', 'test/edge_types_distribution/triple',]

    print_results(results, ACCEPTED_KEYS)
        
        

if __name__ == "__main__":
    main()

Skipping non-numeric metric '_wandb' in run 'qm9-d4-mmff'
Skipping non-numeric metric 'test/sampling/num_edges_hist' in run 'qm9-d4-mmff'
Skipping non-numeric metric 'test/sampling/num_edges_hist_first' in run 'qm9-d4-mmff'
Skipping non-numeric metric 'test/sampling/num_nodes_hist' in run 'qm9-d4-mmff'
Skipping non-numeric metric 'valid/sampling/num_edges_hist' in run 'qm9-d4-mmff'
Skipping non-numeric metric 'valid/sampling/num_edges_hist_first' in run 'qm9-d4-mmff'
Skipping non-numeric metric 'valid/sampling/num_nodes_hist' in run 'qm9-d4-mmff'
Skipping non-numeric metric '_wandb' in run 'qm9-d4-mmff'
Skipping non-numeric metric 'test/sampling/num_edges_hist' in run 'qm9-d4-mmff'
Skipping non-numeric metric 'test/sampling/num_edges_hist_first' in run 'qm9-d4-mmff'
Skipping non-numeric metric 'test/sampling/num_nodes_hist' in run 'qm9-d4-mmff'
Skipping non-numeric metric 'valid/sampling/num_edges_hist' in run 'qm9-d4-mmff'
Skipping non-numeric metric 'valid/sampling/num_edges_hist_fir